# Data Simulation Engine for Restaurant Analytics

## Overview
This notebook generates synthetic datasets for a restaurant analytics system in Kenya, resembling Java House operations. It creates realistic, structured, and reproducible data for enterprise POS and operations analytics.

## Simulation Logic and Assumptions

### Restaurant Master Data
- **restaurant_id**: Unique integer identifier.
- **restaurant_name**: Randomly generated names like 'Java House Nairobi CBD'.
- **location_tier**: One of CBD, Suburb, Highway, Airport. Influences customer base and costs.
- **staff_count**: 10-30 employees, affects staff costs.
- **base_rent**: Monthly rent in KES, varies by location.
- **efficiency_score**: Hidden 0-1 score, affects performance.
- **base_rating**: 3.0-4.9, influences customer attraction.

### Daily Operations Data
- **Customers**: Base on location tier, rating, day-of-week (weekends +20%), month (December +15%), upward trend (1% quarterly), noise (±10%), efficiency score.
- **Avg Order Value**: KES 500-1500, higher in CBD/Airport, with slight inflation.
- **Revenue**: customers * avg_order_value.
- **Food Cost**: 30-45% of revenue, with variability.
- **Staff Cost**: Based on staff_count (KES 25k-80k/month per employee), allocated daily.
- **Rent**: Daily allocation of monthly rent.
- **Operating Expenses**: 10-20% of revenue + noise.
- **Net Profit**: Revenue - all costs.

### Additional Realism
- Random disruptions: Occasional 20-50% customer drops for 3-7 days.
- Inflation: 2% annual increase in costs and prices.
- All values in KES, calibrated to Kenyan market.

## Code Implementation

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

def generate_restaurants(num_restaurants=20):
    restaurants = []
    tiers = ['CBD', 'Suburb', 'Highway', 'Airport']
    tier_rents = {'CBD': (300000, 800000), 'Suburb': (150000, 400000), 'Highway': (100000, 300000), 'Airport': (400000, 1000000)}
    for i in range(1, num_restaurants + 1):
        tier = random.choice(tiers)
        rent_min, rent_max = tier_rents[tier]
        restaurant = {
            'restaurant_id': i,
            'restaurant_name': f'Java House {tier} {i}',
            'location_tier': tier,
            'staff_count': random.randint(10, 30),
            'base_rent': random.randint(rent_min, rent_max),
            'efficiency_score': random.uniform(0.5, 1.0),
            'base_rating': round(random.uniform(3.0, 4.9), 1)
        }
        restaurants.append(restaurant)
    return pd.DataFrame(restaurants)

def generate_daily_data(restaurants_df, start_date, end_date):
    data = []
    current_date = start_date
    while current_date <= end_date:
        for _, rest in restaurants_df.iterrows():
            # Customer base calculation
            base_customers = {'CBD': 250, 'Suburb': 150, 'Highway': 120, 'Airport': 300}[rest['location_tier']]
            rating_boost = (rest['base_rating'] - 3.0) * 50  # Higher rating attracts more
            efficiency_boost = rest['efficiency_score'] * 20
            day_factor = 1.2 if current_date.weekday() >= 5 else 1.0  # Weekend uplift
            month_factor = 1.15 if current_date.month == 12 else 1.0  # December spike
            trend = 1 + (current_date.year - start_date.year + (current_date.month - start_date.month)/12) * 0.01  # 1% quarterly growth
            noise = np.random.normal(1, 0.1)
            
            # Random disruption
            disruption = 1.0
            if random.random() < 0.02:  # 2% chance of disruption
                disruption = random.uniform(0.5, 0.8)  # 20-50% drop
            
            customers = max(0, int((base_customers + rating_boost + efficiency_boost) * day_factor * month_factor * trend * noise * disruption))
            
            # Avg order value
            base_aov = 700 if rest['location_tier'] in ['CBD', 'Airport'] else 600
            inflation = 1 + (current_date.year - start_date.year) * 0.02  # 2% annual inflation
            avg_order_value = round(random.uniform(base_aov * 0.8, base_aov * 1.2) * inflation, 2)
            
            revenue = customers * avg_order_value
            
            # Costs
            food_cost_pct = random.uniform(0.30, 0.45)
            food_cost = revenue * food_cost_pct
            staff_cost = (rest['staff_count'] * random.uniform(25000, 80000) / 30)  # Daily staff cost
            rent_daily = rest['base_rent'] / 30
            op_exp = revenue * random.uniform(0.10, 0.20) + np.random.normal(0, 1000)
            
            net_profit = revenue - food_cost - staff_cost - rent_daily - op_exp
            
            data.append({
                'date': current_date,
                'restaurant_id': rest['restaurant_id'],
                'customers': customers,
                'avg_order_value': avg_order_value,
                'revenue': round(revenue, 2),
                'food_cost': round(food_cost, 2),
                'staff_cost': round(staff_cost, 2),
                'rent': round(rent_daily, 2),
                'operating_expenses': round(op_exp, 2),
                'net_profit': round(net_profit, 2)
            })
        current_date += timedelta(days=1)
    return pd.DataFrame(data)

# Generate data
restaurants = generate_restaurants(20)
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 12, 31)
daily_data = generate_daily_data(restaurants, start_date, end_date)

# Save to CSV
restaurants.to_csv('data/restaurant_master.csv', index=False)
daily_data.to_csv('data/daily_operations.csv', index=False)

print('Data generation complete. Files saved.')